# TFT Geo + Wind — **Paper-vergleichbare Version (1-Stunde-Vorhersage)**

Dies ist eine Variante des Notebooks `TFT_Geo_Wind_Beijing.ipynb`, die **direkt mit dem
Referenz-Paper vergleichbar** sein soll: **Tsokov, Lazarova & Aleksieva-Petrova (2022),**
*Sustainability* **14(9):5104.** Das Paper sagt PM2.5 **eine Stunde** voraus und berichtet
MAE (ug/m3). Deshalb ist hier der **einzige** wesentliche Unterschied zur Hauptversion:

> **`max_prediction_length = 1`** (statt 12) — also 1-Stunde-Vorhersage.

### Ehrliche Einordnung: was bleibt trotzdem unterschiedlich?

Ein echter 1:1-Vergleich ist nicht moeglich; folgende methodische Unterschiede solltet ihr in
der Praesentation nennen:

1. **Ein gemeinsames Modell** ueber alle 12 Stationen vs. im Paper **ein separates Modell je Station**.
2. **Test-Zeitraum**: hier die letzten 14 Tage; das Paper nutzt eine eigene Train/Val/Test-Aufteilung.
3. **Eingabevariablen** und **Fehlwert-Imputation** unterscheiden sich leicht (das Paper waehlt
   seine Variablen per genetischem Algorithmus aus).

Die 1-Stunde-Version macht die MAE-Zahlen also **groessenordnungsmaessig vergleichbar** — nicht exakt.
Der *faire* Vergleich (Paper-Testjahr, Modell je Station) steht in `TFT_PaperReplikat_ProStation`.

## 0. Installation

Einmalig ausfuehren. `pytorch-forecasting` ist versionsempfindlich — deshalb passende Versionen zusammen installieren.

In [ ]:
# Einmalig ausfuehren (dann kannst du die Zelle auskommentieren).
%pip install -q "torch>=2.0" "lightning>=2.0" "pytorch-forecasting>=1.0" pandas numpy matplotlib
print("Installation fertig. Bitte ggf. den Kernel neu starten (Kernel -> Restart).")

## 1. Bibliotheken importieren

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

pl.seed_everything(42)          # Reproduzierbarkeit

# Tensor-Cores der GPU besser nutzen -> schnelleres Training, minimaler Genauigkeitsverlust
torch.set_float32_matmul_precision("medium")

print("torch:", torch.__version__)
print("GPU verfuegbar:", torch.cuda.is_available())

## 2. Daten laden

Beijing Multi-Site Air Quality (12 Stationen, 2013-2017). Zuerst Download-Versuch von UCI,
sonst synthetischer Ersatz mit identischer Struktur.

In [ ]:
import io, zipfile, urllib.request, os

DATA_DIR = "beijing_data"
UCI_URL = "https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip"

STATIONS = ["Aotizhongxin","Changping","Dingling","Dongsi","Guanyuan","Gucheng",
            "Huairou","Nongzhanguan","Shunyi","Tiantan","Wanliu","Wanshouxigong"]

def try_download_real_data():
    try:
        print("Lade UCI-Datensatz herunter ...")
        raw = urllib.request.urlopen(UCI_URL, timeout=30).read()
        z = zipfile.ZipFile(io.BytesIO(raw))
        os.makedirs(DATA_DIR, exist_ok=True)
        for name in z.namelist():
            if name.endswith(".zip"):
                inner = zipfile.ZipFile(io.BytesIO(z.read(name)))
                inner.extractall(DATA_DIR)
            elif name.endswith(".csv"):
                z.extract(name, DATA_DIR)
        print("Download OK.")
        return True
    except Exception as e:
        print("Download fehlgeschlagen:", e)
        return False

def load_real_csvs():
    import glob
    files = glob.glob(os.path.join(DATA_DIR, "**", "PRSA_Data_*.csv"), recursive=True)
    if len(files) < 12:
        return None
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

def make_synthetic():
    print(">>> Nutze SYNTHETISCHE Demo-Daten (nur zum Testen!) <<<")
    rng = np.random.default_rng(0)
    frames = []
    hours = pd.date_range("2013-03-01", periods=24*40, freq="h")
    for i, st in enumerate(STATIONS):
        n = len(hours)
        daily = 30*np.sin(2*np.pi*hours.hour/24)
        frames.append(pd.DataFrame({
            "year":hours.year,"month":hours.month,"day":hours.day,"hour":hours.hour,
            "PM2.5":(80+daily+rng.normal(0,15,n)).clip(1),
            "PM10":(100+daily+rng.normal(0,20,n)).clip(1),
            "SO2":rng.normal(15,5,n).clip(0),"NO2":rng.normal(50,15,n).clip(0),
            "O3":rng.normal(60,20,n).clip(0),
            "TEMP":13+10*np.sin(2*np.pi*hours.dayofyear/365)+rng.normal(0,2,n),
            "DEWP":rng.normal(2,5,n),"RAIN":0.0,
            "wd":rng.choice(["N","NE","E","SE","S","SW","W","NW"],n),
            "WSPM":rng.normal(2,1,n).clip(0),"station":st}))
    return pd.concat(frames, ignore_index=True)

raw = None
if try_download_real_data():
    raw = load_real_csvs()
if raw is None:
    raw = make_synthetic()

print("Roh-Datensatz (alle Stationen gestapelt):", raw.shape)
raw.head()

## 3. Vorverarbeitung: Zeitstempel, `time_idx`, Wind, NaN-Auffuellen

Wie in der Hauptversion. `PM2.5` wird zu `PM25` umbenannt (pytorch-forecasting verbietet Punkte in Spaltennamen).

In [ ]:
df = raw.copy()

df["datetime"] = pd.to_datetime(df[["year","month","day","hour"]])
df = df.sort_values(["station","datetime"]).reset_index(drop=True)
df["time_idx"] = ((df["datetime"] - df["datetime"].min()).dt.total_seconds() // 3600).astype(int)

df["hour"]      = df["datetime"].dt.hour
df["dayofweek"] = df["datetime"].dt.dayofweek

df["wd"] = df["wd"].astype(str).fillna("unknown").replace("nan","unknown")

num_cols = ["PM2.5","PM10","SO2","NO2","O3","TEMP","DEWP","RAIN","WSPM"]
num_cols = [c for c in num_cols if c in df.columns]
df[num_cols] = (df.groupby("station")[num_cols]
                  .apply(lambda g: g.ffill().bfill())
                  .reset_index(level=0, drop=True))
df[num_cols] = df[num_cols].fillna(0)

# Punkte in Spaltennamen sind verboten -> PM2.5 zu PM25
df = df.rename(columns={"PM2.5": "PM25"})
num_cols = ["PM25" if c == "PM2.5" else c for c in num_cols]

print("Fehlende Werte nach dem Auffuellen:", int(df[num_cols].isna().sum().sum()))
df[["datetime","station","time_idx","PM25","WSPM","wd","hour","dayofweek"]].head()

## 4. Geo-Embedding: Koordinaten je Station laden

Koordinaten aus `data/stations_geo.csv` (offizielle Werte aller 12 Stationen aus
Tsokov et al. 2022, Table 1, verifiziert).

In [ ]:
import os

geo_path = os.path.join("..", "data", "stations_geo.csv")
coord_df = pd.read_csv(geo_path)[["station", "lon", "lat"]]

df = df.merge(coord_df, on="station", how="left")
for c in ["lon", "lat"]:
    lo, hi = df[c].min(), df[c].max()
    df[c + "_norm"] = (df[c] - lo) / (hi - lo + 1e-9)

fehlend = df[df["lon"].isna()]["station"].unique()
assert len(fehlend) == 0, f"Es fehlen Koordinaten fuer: {fehlend}"
print("Koordinaten gemerged. Stationen gesamt:", df["station"].nunique())

## 5. `TimeSeriesDataSet` — **1-Stunde-Vorhersage (Paper-vergleichbar)**

Einziger inhaltlicher Unterschied zur Hauptversion: **`max_prediction_length = 1`**.
Damit sagt das Modell — wie das Paper — die **naechste Stunde** voraus.
Test sind rollierende 1h-Fenster ueber die letzten 14 Tage je Station.

In [ ]:
max_encoder_length    = 48   # Stunden Vergangenheit, die das Modell sieht
max_prediction_length = 1    # <<< PAPER-VERGLEICH: 1 Stunde voraus (Hauptversion nutzt 12)

target = "PM25"

# Zeitlicher Train/Test-Split: letzte 14 Tage als Test-Zeitraum
test_days   = 14
test_length = test_days * 24
training_cutoff = df["time_idx"].max() - test_length

training = TimeSeriesDataSet(
    df[df["time_idx"] <= training_cutoff],
    time_idx="time_idx",
    target=target,
    group_ids=["station"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    static_categoricals=["station"],
    static_reals=["lon_norm", "lat_norm"],     # Geo-Embedding

    time_varying_known_reals=["time_idx", "hour", "dayofweek"],
    time_varying_unknown_reals=[
        "PM25", "PM10", "SO2", "NO2", "O3",
        "TEMP", "DEWP", "RAIN",
        "WSPM",                                 # Wind
    ],
    time_varying_unknown_categoricals=["wd"],

    target_normalizer=GroupNormalizer(groups=["station"]),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training, df,
    min_prediction_idx=training_cutoff + 1,
    stop_randomization=True,
)

print(f"Vorhersage-Horizont: {max_prediction_length} Stunde(n) (Paper-vergleichbar)")
print(f"Test-Zeitraum: letzte {test_days} Tage je Station")
print("Trainings-Samples:", len(training), "| Test-Samples:", len(validation))

## 6. DataLoader und Modell

In [ ]:
batch_size = 128   # bei "CUDA out of memory" auf 64 reduzieren
train_dataloader = training.to_dataloader(train=True,  batch_size=batch_size, num_workers=0)
val_dataloader   = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=0)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=64,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=32,
    loss=QuantileLoss(),
    optimizer="adam",
    reduce_on_plateau_patience=4,
)
print(f"Modell erstellt. Anzahl Parameter: {tft.size()/1e3:.1f}k")

## 7. Training

In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint

early_stop = EarlyStopping(monitor="val_loss", patience=5, mode="min")
checkpoint = ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1)

trainer = pl.Trainer(
    max_epochs=30,
    accelerator="auto",
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[early_stop, checkpoint],
    enable_progress_bar=True,
    logger=False,
)

trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
print("Training fertig.")
print("Bestes Modell gespeichert unter:", checkpoint.best_model_path)

## 8. Bestes Modell laden

In [ ]:
best_model_path = checkpoint.best_model_path
if best_model_path:
    best_model = TemporalFusionTransformer.load_from_checkpoint(best_model_path)
else:
    best_model = tft

raw_predictions = best_model.predict(val_dataloader, mode="raw", return_x=True)
print("Vorhersagen berechnet.")

## 8b. Fehlerkennzahlen je Station (1h-Vorhersage) + CSV-Export

Gleiche Kennzahlen wie in der Hauptversion, aber fuer den **1-Stunden-Horizont** — die
**MAE-Spalte** ist die Zahl, die ihr mit dem Paper vergleicht.

In [ ]:
import os
import numpy as np
import pandas as pd

pred_s = best_model.predict(val_dataloader, mode="prediction", return_y=True, return_index=True)
yp = pred_s.output.detach().cpu().numpy()
yt = pred_s.y[0].detach().cpu().numpy()
stations = pred_s.index["station"].to_numpy()

# Naive 24h-Baseline je Station (fuer MASE)
SEASON = 24
train_df = df[df["time_idx"] <= training_cutoff]
scale = {}
for st, g in train_df.sort_values("time_idx").groupby("station"):
    scale[st] = g[target].diff(SEASON).abs().dropna().mean()

rows = []
for st in np.unique(stations):
    m = stations == st
    a = yp[m].reshape(-1); b = yt[m].reshape(-1)
    mae   = np.mean(np.abs(a - b))
    rmse  = np.sqrt(np.mean((a - b) ** 2))
    mape  = np.mean(np.abs(a - b) / np.clip(np.abs(b), 1, None)) * 100
    naive = scale.get(st, np.nan)
    mase  = mae / naive if naive else np.nan
    rows.append({"Station": st, "n_Fenster": int(m.sum()),
                 "MAE": mae, "MAE_naive24h": naive, "MASE": mase,
                 "RMSE": rmse, "MAPE_%": mape})

metrics_df = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)

a_all, b_all = yp.reshape(-1), yt.reshape(-1)
naive_mean = float(np.mean([scale[s] for s in np.unique(stations)]))
gesamt = {"Station": "GESAMT", "n_Fenster": int(len(stations)),
          "MAE": np.mean(np.abs(a_all - b_all)), "MAE_naive24h": naive_mean,
          "MASE": metrics_df["MASE"].mean(),
          "RMSE": np.sqrt(np.mean((a_all - b_all) ** 2)),
          "MAPE_%": metrics_df["MAPE_%"].mean()}
metrics_df = pd.concat([metrics_df, pd.DataFrame([gesamt])], ignore_index=True)
metrics_df = metrics_df[["Station","n_Fenster","MAE","MAE_naive24h","MASE","RMSE","MAPE_%"]]

print("Fehlerkennzahlen je Station (1h-Vorhersage):\n")
print(metrics_df.round(3).to_string(index=False))

out_path = os.path.join("..", "data", "ergebnis_tft_geo_metriken_1h.csv")
metrics_df.round(4).to_csv(out_path, index=False)
print("\nGespeichert unter:", out_path)

## 8c. Direkter Vergleich mit dem Paper (Dongsi, Wanliu, Changping)

Die Paper-MAE-Werte sind eingetragen (Quelle: **Tsokov et al. 2022, Tables 3-5**; bester der drei
Experimentierlaeufe je Station, jeder ueber 10 Trainingslaeufe gemittelt).

> Denkt an die Einordnung aus der Titelzelle: hier **ein gemeinsames Modell** ueber alle Stationen
> und ein **14-Tage-Testfenster** statt des Paper-Testjahres. Dieser Vergleich ist daher nur
> **indikativ** — der *faire* Vergleich steht im Notebook `TFT_PaperReplikat_ProStation`.

In [ ]:
# Paper-MAE (ug/m3), Quelle: Tsokov et al. 2022, Tables 3-5
# (bester der 3 Experimentierlaeufe, jeder ueber 10 Trainingslaeufe gemittelt)
paper_mae = {
    "Dongsi":    17.87,   # Table 3 (Bereich 17.9-19.1)
    "Wanliu":    15.37,   # Table 4 (Bereich 15.4-16.8)
    "Changping": 15.16,   # Table 5 (Bereich 15.2-16.1)
}

vgl = metrics_df[metrics_df["Station"].isin(paper_mae.keys())][["Station", "MAE"]].copy()
vgl = vgl.rename(columns={"MAE": "MAE_unser_Modell"})
vgl["MAE_Paper"]  = vgl["Station"].map(paper_mae)
vgl["Differenz"]  = vgl["MAE_unser_Modell"] - vgl["MAE_Paper"]

print("Vergleich 1h-PM2.5-Vorhersage, MAE in ug/m3 (niedriger = besser):\n")
print(vgl.round(2).to_string(index=False))
print("\nHinweis: gemeinsames Modell + 14-Tage-Test -> nur indikativ (siehe Per-Station-Notebook).")

# Balkendiagramm
plot_vgl = vgl.dropna(subset=["MAE_Paper"])
if len(plot_vgl):
    x = np.arange(len(plot_vgl)); w = 0.4
    plt.figure(figsize=(7, 4))
    plt.bar(x - w/2, plot_vgl["MAE_unser_Modell"], w, label="Unser TFT")
    plt.bar(x + w/2, plot_vgl["MAE_Paper"],        w, label="Paper (Tsokov 2022)")
    plt.xticks(x, plot_vgl["Station"])
    plt.ylabel("MAE (ug/m3)"); plt.title("1h-PM2.5-Vorhersage: unser Modell vs. Paper")
    plt.legend(); plt.tight_layout(); plt.show()

## 9. Variablen-Wichtigkeit — Wind und Geo

Prozent-Rangliste innerhalb jeder Gruppe. `WSPM` sollte bei den Encoder-Variablen, `lon`/`lat`
bei den statischen Variablen auftauchen.

In [ ]:
import numpy as np

interpretation = best_model.interpret_output(raw_predictions.output, reduction="sum")
best_model.plot_interpretation(interpretation)
plt.show()

def zeige_wichtigkeit(titel, namen, werte):
    werte = np.asarray(werte, dtype=float)
    summe = werte.sum()
    prozent = 100 * werte / summe if summe > 0 else werte
    print(titel)
    for n, p in sorted(zip(namen, prozent), key=lambda t: -t[1]):
        print(f"  {n:22s}: {p:5.1f} %")
    print()

zeige_wichtigkeit("Encoder-Variablen (Vergangenheit, u.a. WSPM = Wind):",
                  best_model.encoder_variables,
                  interpretation["encoder_variables"].detach().cpu().numpy())

zeige_wichtigkeit("Statische (Stations-)Variablen (Geo: lon_norm / lat_norm):",
                  best_model.static_variables,
                  interpretation["static_variables"].detach().cpu().numpy())